# Databricks Project



In [0]:
# Costumized data frame
data=[(1,'aa',30),(2,'bb',40),(3,'cc',50)]
my_schema="id INT,name STRING,marks INT"
df=spark.createDataFrame(data,schema=my_schema)


In [0]:
df.display()

In [0]:
# With this step the databricks can be allow to read data from the datalake (Azure)
# Copy the above key values in the given places

# Secret dbutils

service_credential = dbutils.secrets.get(
    scope='utkarshlscope',
    key='applicationid-secret'
)




spark.conf.set("fs.azure.account.auth.type.datalakeutkarshl.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.datalakeutkarshl.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.datalakeutkarshl.dfs.core.windows.net", "df4507de-c603-441f-a45d-906b68f4e773")
spark.conf.set("fs.azure.account.oauth2.client.secret.datalakeutkarshl.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.datalakeutkarshl.dfs.core.windows.net", "https://login.microsoftonline.com/037e12b0-3666-42ef-9548-b509e4df25c5/oauth2/token")

### Access Data


### DB Untilities

**dbutils.fs()**

In [0]:
dbutils.fs.ls("abfss://source@datalakeutkarshl.dfs.core.windows.net/")


**dbutils.widgets**

In [0]:
# Dropdown widgets
dbutils.widgets.dropdown(
    "item_type",
    "All",
    [
        "All",
        "Baking Goods",
        "Breads",
        "Breakfast",
        "Canned",
        "Dairy",
        "Frozen Foods",
        "Fruits and Vegetables",
        "Hard Drinks",
        "Health and Hygiene",
        "Household",
        "Meat",
        "Others",
        "Seafood",
        "Snack Foods",
        "Soft Drinks",
        "Starchy Foods"
    ],
    "Select Item Type"
)


In [0]:
# Get the value of the user selected
item_type = dbutils.widgets.get("item_type")
print(item_type)

In [0]:
# Dropdown Outlet Type

dbutils.widgets.dropdown(
    "outlet_type",
    "All",
    [
        "All",
        "Grocery Store",
        "Supermarket Type1",
        "Supermarket Type2",
        "Supermarket Type3"
    ],
    "Select Outlet Type"
)

In [0]:
outlet_type = dbutils.widgets.get("outlet_type")
print(outlet_type)

**dbutils.secrets**


In [0]:
#dbutils.secrets.list(scope='utkarshlscope')

In [0]:
#dbutils.secrets.get(scope ='utkarshlscope', key = 'applicationid-secret')

**Data Reading**


In [0]:
df_sales = spark.read.format('csv')\
              .option('header',True)\
                .option('inferSchema', True)\
                    .load('abfss://source@datalakeutkarshl.dfs.core.windows.net/')
                  
            

In [0]:
df_sales.display()

**Schema and Data Types**

In [0]:
df_sales.printSchema()

**Chesk for Null Values**


In [0]:
from pyspark.sql.functions import col, sum

df_sales.select([
    sum(col(column).isNull().cast("int")).alias(column)
    for column in df_sales.columns
]).show()

**Check for duplicate rows**

In [0]:
total_rows = df_sales.count()

unique_rows = df_sales.dropDuplicates().count()

duplicate_rows = total_rows - unique_rows

print(f"Total rows: {total_rows}")
print(f"Unique rows: {unique_rows}")
print(f"Duplicate rows: {duplicate_rows}")

**Check Unique values in Categorical Columns**

In [0]:
from pyspark.sql.functions import countDistinct

for column in df_sales.columns:
    df_sales.select(
        countDistinct(column).alias(f"Unique values in {column}")
    ).show()

In [0]:
# Data qaulity problem in Item Fat Content

df_sales.select("Item_Fat_Content").distinct().show()

In [0]:
df_sales.select("Item_Type").distinct().show()

In [0]:
df_sales.select("Outlet_Size").distinct().show()

In [0]:
df_sales.select("Outlet_Location_Type").distinct().show()

**Numerical column summaries**


In [0]:
df_sales.describe().show()

**Check the count of the Item Visibility stating 0**

In [0]:
df_sales.filter(
    df_sales.Item_Visibility == 0
).count()

In [0]:
df_sales.filter(
    df_sales.Item_Visibility == 0
).show(10)

**Investigate missing values**

In [0]:
from pyspark.sql.functions import col, sum

df_sales.groupBy("Item_Identifier")\
    .agg(
        sum(col("Item_Weight").isNull().cast("int")).alias("Missing_Weight")
    )\
    .orderBy("Missing_Weight", ascending=False)\
    .show()

In [0]:
df_sales.groupBy("Outlet_Identifier")\
    .agg(
        sum(col("Outlet_Size").isNull().cast("int")).alias("Missing_Outlet_Size")
    )\
    .orderBy("Missing_Outlet_Size", ascending=False)\
    .show()

Key observation: Data Quality assessment

Item fat conetent is not standardised, the values showing variations like Low Fat, LF, low fat, Regular and reg. There are only two content which are Low Fat and Regular, needs to be standardised. These represent inconsistent labels for the same business categories. This is a confirmed data quality issue.


Item visibility has a minimum value of 0, but there no confirmation whether zero is valid, but this is not an error either. There may be no recorded display on the product. Or there be null instead of 0.The  zero values cannot be confirmed from the available dataset and requires further investigation before any transformation is applied.

Null values/ missing values in Item weight and Outlet size which need to investigate.


